In [1]:
import json
import cv2
import motmetrics as mm
from pathlib import Path
from ultralytics import YOLO

model = YOLO("../runs/runs/baseline_yolo11s_e100/runs/kaggle_yolo11s_optimized/weights/best.pt")

video_path = "../Anti-UAV-RGBT/test/20190926_134054_1_1/infrared.mp4"
gt_path = "../Anti-UAV-RGBT/test/20190926_134054_1_1/infrared.json"

assets_dir = Path("../assets")
assets_dir.mkdir(exist_ok=True)
out_path = assets_dir / "20190926_134054_1_1_infrared_tracked.mp4"

with open(gt_path) as f:
    gt = json.load(f)
gt_exist, gt_rect = gt["exist"], gt["gt_rect"]

In [ ]:

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
cap.release()

writer = None
per_frame_preds = []

for result in model.track(
    source=video_path,
    tracker="../bytetrack/bytetrack.yaml",
    stream=True,
    persist=True,
    verbose=False,
):
    annotated = result.plot()
    if writer is None:
        h, w = annotated.shape[:2]
        writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    writer.write(annotated)

    frame_preds = []
    if result.boxes is not None and result.boxes.id is not None:
        ids = result.boxes.id.int().tolist()
        for tid, (x1, y1, x2, y2) in zip(ids, result.boxes.xyxy.tolist()):
            frame_preds.append((tid, [x1, y1, x2 - x1, y2 - y1]))
    per_frame_preds.append(frame_preds)

writer.release()
print(f"Saved demo video to {out_path} ({len(per_frame_preds)} frames)")

Saved demo video to ../assets/20190926_134054_1_1_infrared_tracked.mp4 (1000 frames)


## Tracking metrics

Standard MOT metrics (MOTA, IDF1, ID switches) via `motmetrics`, computed frame-by-frame
against the ground truth `gt_rect`/`exist` in `infrared.json`.

In [3]:
assert len(per_frame_preds) == len(gt_exist), (
    f"frame count mismatch: {len(per_frame_preds)} predicted vs {len(gt_exist)} ground truth"
)

acc = mm.MOTAccumulator(auto_id=True)
for t, preds in enumerate(per_frame_preds):
    gt_ids, gt_boxes = ([1], [gt_rect[t]]) if gt_exist[t] else ([], [])
    hyp_ids = [tid for tid, _ in preds]
    hyp_boxes = [box for _, box in preds]
    dists = mm.distances.iou_matrix(gt_boxes, hyp_boxes, max_iou=0.5)
    acc.update(gt_ids, hyp_ids, dists)

mh = mm.metrics.create()
summary = mh.compute(acc, metrics=["mota", "idf1", "num_switches"], name="infrared")
print(summary)

           mota     idf1  num_switches
infrared  0.933  0.72969            10


## mSA (mean State Accuracy)

The Anti-UAV benchmark's own metric:

$$acc = \sum_{t=1}^{T} \frac{IoU_t \cdot \delta(v_t>0) + p_t \cdot (1-\delta(v_t>0))}{T} - 0.2 \times \left(\sum_{t=1}^{T^*} \frac{p_t \cdot \delta(v_t>0)}{T^*}\right)^{0.3}$$

`v_t` = ground-truth visibility (`exist`), `p_t` = 1 iff the predicted box is empty that frame,
`T` = total frames, `T*` = frames where the target is visible in the ground truth.

In [4]:
def iou_xywh(box_a, box_b):
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def state_accuracy(gt_exist, gt_rect, per_frame_preds):
    T = len(gt_exist)
    T_star = sum(1 for v in gt_exist if v > 0)
    total = 0.0
    penalty_sum = 0.0
    for t in range(T):
        visible = gt_exist[t] > 0
        has_pred = len(per_frame_preds[t]) > 0
        p_t = 0 if has_pred else 1
        if visible:
            iou_t = max((iou_xywh(gt_rect[t], box) for _, box in per_frame_preds[t]), default=0.0)
            total += iou_t
            penalty_sum += p_t
        else:
            total += p_t
    penalty = 0.2 * ((penalty_sum / T_star) ** 0.3) if T_star else 0.0
    return total / T - penalty


msa = state_accuracy(gt_exist, gt_rect, per_frame_preds)
print(f"mSA: {msa:.4f}")

mSA: 0.7440
